In [1]:
import json
import os
import warnings
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
import time
import math

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Install timm jika belum ada
try:
    import timm
except ImportError:
    os.system('pip install timm -q')
    import timm

warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def auto_log_experiment(env_globals: dict, topic: str, model: str, objective: str, dataset: str, owner: str) -> None:
    """
    Fungsi cerdas untuk memindai variabel metrik di notebook secara otomatis 
    dan mencetaknya sebagai payload registri.
    """
    expected_metrics = {
        "accuracy": ["accuracy", "acc", "acc_val", "accuracy_score"],
        "f1": ["f1", "f1_score", "f1_val"],
        "precision": ["precision", "prec_val"],
        "recall": ["recall", "rec_val"],
        "rmse": ["rmse", "rmse_val"],
        "mae": ["mae", "mae_val"],
        "r2_score": ["r2", "r2_score", "r2_val"]
    }
    
    found_metrics = {}
    
    for metric_key, possible_vars in expected_metrics.items():
        for var_name in possible_vars:
            if var_name in env_globals:
                val = env_globals[var_name]
                if isinstance(val, (int, float)):
                    found_metrics[metric_key] = float(val)
                break

    if not found_metrics:
        print("PERINGATAN: Tidak ada variabel metrik numerik yang terdeteksi di notebook.")
        return

    metadata = {
        "topic": topic,
        "model": model,
        "objective": objective,
        "dataset": dataset,
        "owner": owner,
        "metrics": found_metrics
    }
    
    print("\n===REGISTRY_PAYLOAD_START===")
    print(json.dumps(metadata))
    print("===REGISTRY_PAYLOAD_END===")

# ==========================================
# KONFIGURASI AWAL
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

base_dir   = "/kaggle/input/datasets/sonymikio/skin-undertone"
train_dir  = os.path.join(base_dir, "train")
valid_dir  = os.path.join(base_dir, "valid")
test_dir   = os.path.join(base_dir, "test")
output_dir = "/kaggle/working"

NUM_EPOCHS     = 95
BATCH_SIZE     = 16
IMG_SIZE       = 224
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 0.05
LABEL_SMOOTH   = 0.1
MIXUP_ALPHA    = 0.2
WARMUP_EPOCHS  = 5
PATIENCE       = 10
DROPOUT_RATE   = 0.3
GRAD_CLIP      = 1.0

MODEL_PATH = os.path.join(output_dir, "skin_undertone_convnextv2.pth")
class_names = ["cool", "neutral", "warm"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# ==========================================
# AUGMENTASI
# ==========================================
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
    transforms.RandomGrayscale(p=0.02),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + (self.base_lr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr

# ==========================================
# ARSITEKTUR MODEL
# ==========================================
class SkinUndertoneModel(nn.Module):
    def __init__(self, num_classes=3, dropout_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'convnextv2_tiny.fcmae_ft_in22k_in1k',
            pretrained=True,
            num_classes=0
        )
        num_features = self.backbone.num_features
        for name, param in self.backbone.named_parameters():
            if 'stages.0' in name or 'stages.1' in name:
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 256),
            nn.GELU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        out = self.classifier(features)
        return out

# ==========================================
# EVALUASI TEST SET
# ==========================================
def evaluate_test_set():
    if not os.path.exists(test_dir):
        print("\n⚠️ Folder test tidak ditemukan, melewati evaluasi test.")
        return None
    if not os.path.exists(MODEL_PATH):
        print("\n⚠️ File model belum ada.")
        return None

    test_dataset = datasets.ImageFolder(test_dir, transform=eval_transforms)
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )

    model = SkinUndertoneModel(num_classes=len(class_names), dropout_rate=DROPOUT_RATE).to(device)
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    criterion = nn.CrossEntropyLoss()
    num_classes = len(class_names)

    test_loss = 0.0
    test_correct = 0
    test_total = 0
    confusion = [[0] * num_classes for _ in range(num_classes)]

    print("\n--- Memulai Evaluasi pada Test Set ---")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            _, preds = torch.max(outputs, 1)
            test_loss += loss.item() * inputs.size(0)
            test_correct += (preds == labels).sum().item()
            test_total += inputs.size(0)

            for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                confusion[t][p] += 1

    test_acc = test_correct / test_total * 100
    
    # Hitung Macro Precision, Recall, F1
    precisions, recalls, f1s = [], [], []
    for i in range(num_classes):
        tp = confusion[i][i]
        fp = sum(confusion[r][i] for r in range(num_classes)) - tp
        fn = sum(confusion[i][c] for c in range(num_classes)) - tp
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
        
    macro_prec = sum(precisions) / num_classes * 100
    macro_rec = sum(recalls) / num_classes * 100
    macro_f1 = sum(f1s) / num_classes * 100

    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Macro Precision: {macro_prec:.2f}%")
    print(f"Macro Recall: {macro_rec:.2f}%")
    print(f"Macro F1-Score: {macro_f1:.2f}%")

    return {
        "accuracy": test_acc,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1
    }

# ==========================================
# PELATIHAN MODEL
# ==========================================
def train_model():
    if not os.path.exists(train_dir) or not os.path.exists(valid_dir):
        print("❌ Dataset tidak ditemukan. Pastikan path dataset benar di Kaggle.")
        return None

    train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
    valid_dataset = datasets.ImageFolder(valid_dir, transform=eval_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    targets = train_dataset.targets
    class_counts = [targets.count(i) for i in range(len(class_names))]
    total_samples = sum(class_counts)
    class_weights = [total_samples / (len(class_names) * c) for c in class_counts]
    class_weights_tensor = torch.FloatTensor(class_weights).to(device)

    model = SkinUndertoneModel(num_classes=len(class_names), dropout_rate=DROPOUT_RATE).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTH)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
    scheduler = WarmupCosineScheduler(optimizer, warmup_epochs=WARMUP_EPOCHS, total_epochs=NUM_EPOCHS, min_lr=1e-6)

    best_val_acc = 0.0
    no_improve_count = 0

    for epoch in range(NUM_EPOCHS):
        lr = scheduler.step(epoch)
        model.train()
        
        # --- TRAINING LOOP (Disederhanakan untuk visual log di sini) ---
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            use_mixup = (MIXUP_ALPHA > 0) and (np.random.random() > 0.3)
            
            if use_mixup:
                inputs, labels_a, labels_b, lam = mixup_data(inputs, labels, MIXUP_ALPHA)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            if use_mixup:
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        # --- VALIDATION LOOP ---
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                v_correct += (preds == labels).sum().item()
                v_total += inputs.size(0)

        val_acc = v_correct / v_total * 100
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve_count = 0
            torch.save({'model_state_dict': model.state_dict()}, MODEL_PATH)
        else:
            no_improve_count += 1
            if no_improve_count >= PATIENCE:
                print(f"⚠️ Early Stopping di Epoch {epoch+1}")
                break

    # Panggil evaluasi setelan akhir
    return evaluate_test_set()

# ==========================================
# EKSEKUSI UTAMA (MAIN)
# ==========================================
if __name__ == "__main__":
    print("=== MULAI TRAINING ===")
    final_metrics = train_model()
    
    # Otomatis log metrik jika training berhasil dan mengembalikan dict final_metrics
    if final_metrics:
        # Deklarasi variabel global agar bisa dipindai oleh fungsi `globals()`
        accuracy = final_metrics["accuracy"]
        f1_score = final_metrics["macro_f1"]
        precision = final_metrics["macro_precision"]
        recall = final_metrics["macro_recall"]
        
        auto_log_experiment(
            env_globals=globals(),
            topic="topic-b",
            model="ConvNeXt V2",
            objective="Menggunakan", 
            dataset="skin-undertone",
            owner="Egyanta Bareka Sebayang"
        )

🖥️  Device: cuda
   GPU: Tesla T4
=== MULAI TRAINING ===


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1/95 | Val Acc: 33.33%
Epoch 2/95 | Val Acc: 41.83%
Epoch 3/95 | Val Acc: 32.68%
Epoch 4/95 | Val Acc: 47.06%
Epoch 5/95 | Val Acc: 56.21%
Epoch 6/95 | Val Acc: 52.94%
Epoch 7/95 | Val Acc: 48.37%
Epoch 8/95 | Val Acc: 56.21%
Epoch 9/95 | Val Acc: 60.78%
Epoch 10/95 | Val Acc: 50.98%
Epoch 11/95 | Val Acc: 54.90%
Epoch 12/95 | Val Acc: 53.59%
Epoch 13/95 | Val Acc: 55.56%
Epoch 14/95 | Val Acc: 46.41%
Epoch 15/95 | Val Acc: 55.56%
Epoch 16/95 | Val Acc: 52.94%
Epoch 17/95 | Val Acc: 56.86%
Epoch 18/95 | Val Acc: 46.41%
Epoch 19/95 | Val Acc: 45.75%
⚠️ Early Stopping di Epoch 19

--- Memulai Evaluasi pada Test Set ---
Test Accuracy: 56.96%
Macro Precision: 55.03%
Macro Recall: 58.78%
Macro F1-Score: 55.70%

===REGISTRY_PAYLOAD_START===
{"topic": "topic-b", "model": "ConvNeXt V2", "objective": "Menggunakan", "dataset": "skin-undertone", "owner": "Egyanta Bareka Sebayang", "metrics": {"accuracy": 56.9620253164557, "f1": 55.70111287758347, "precision": 55.03105590062112, "recall": 58